# Notebook 07 — Adaptation

**The system improves itself by reading its own traces and rewriting its own context — lessons, prompts, skills, tools.**

You'll start with the cheapest lever — an agent **writing a lesson learned from a real
failure trace back into its identity file** — then watch a weak prompt get *measurably*
better across epochs by reflecting on its own failures, and watch what happens when you
remove the validation gate (spoiler: "self-improving" becomes "self-drifting"). Then the
same engine optimizes a **skill doc** with bounded edits, and finally graduates a
repeated trace into a **reusable tool**.

> Where this sits in the five moves: adaptation is the agentic loop pointed at the
> agent's *own context* — it rewrites what you **STORE** (the lesson, the prompt, the
> skill, the tool set). The held-out gate is the discipline that keeps a rewrite an
> *improvement* and not a *poisoning* source injected into every future call.

Everything here is **offline-safe, gated, and a readable diff**. The implementations are
faithful **toy** versions of real papers (Reflexion, GEPA, SkillOpt, ReUseIt/Autobrowse),
clearly labeled — we show the *mechanism*, not paper-scale numbers.

## Setup

One import wires the model, a token counter, and the data path. We select the inline plotting backend up front (before the sync↔async bridge spins up a background loop), then pull in the themed chart helpers so every visual below is a one-liner.

In [ ]:
from pathlib import Path
import json, random, importlib, sys
from textwrap import dedent

from roadshow import setup, invoke_sync, run_sync, Message

# Select the inline backend eagerly, on the main thread, BEFORE the async bridge starts
# a background loop — otherwise matplotlib's lazy backend import can race that thread.
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

# Themed chart helpers — every visual in this notebook is one call into these.
from roadshow_viz import lines, scatter, compare

# arcrun: used by the trace -> tool graduation cell near the end.
from arcrun import run, Tool, ToolContext, StaticProvider

model, ask, ntok, DATA = setup()

**Name the task and split the data.** We classify scrap-log lines into seven codes, with a 40-line train set and a 20-line held-out set — the held-out set is what every gate below checks against.

In [ ]:
# Lesson aliases for this notebook.
SCRAP       = DATA / "scrap"
ADAPT       = DATA / "adaptation"
SCRIPTS_OUT = DATA.parent / "scripts"      # a graduated tool will be written here

# The task: classify scrap-log lines into 7 codes (reused from the harness section).
CODES = ["DIM_OOT", "SURF_DEF", "MAT_FLAW", "MACH_SET", "TOOL_WEAR", "OP_ERR", "CONTAM"]
LINES = [json.loads(l) for l in (SCRAP / "scrap_lines.jsonl").read_text().splitlines() if l.strip()]
random.seed(7)
random.shuffle(LINES)
train   = LINES[:40]      # 40 train
heldout = LINES[40:60]    # 20 held-out validation

print(f"task: classify {len(LINES)} scrap lines into {len(CODES)} codes")
print(f"split: {len(train)} train / {len(heldout)} held-out val")

## ▶ Control cell — the only cell you edit

Every knob for this notebook lives here. Change a value, then re-run this cell and the
cells below it. Each knob:

- **`OPTIMIZER`** — `"gepa"` (optimize a prompt) or `"skillopt"` (optimize a skill doc).
- **`EPOCHS`** — how many reflect → mutate → gate rounds to run.
- **`USE_HELDOUT_GATE`** — accept an edit only if held-out score improves. **Flip it to
  `False` to *see* overfitting**: train keeps rising while held-out drifts down.
- **`SEED_PROMPT`** — the deliberately weak starting prompt the optimizer improves.
- **`SEED`** — fixes the shuffle/sampling so a run is reproducible.

In [2]:
# ── CONTROL CELL ──
OPTIMIZER        = "gepa"        # "gepa" | "skillopt"
EPOCHS           = 5
USE_HELDOUT_GATE = True          # flip False to SEE overfitting / drift
SEED_PROMPT      = "Classify each line into a scrap code."   # deliberately weak
SEED             = 7

# If the model is reachable we run the live optimizer; otherwise we fall back to the
# recorded run so every visual still renders (see the try/except gate below).
try:
    model, ask, ntok, DATA = setup()
    MODEL_AVAILABLE = True
except Exception as e:
    MODEL_AVAILABLE = False
    print(f"[offline] model unavailable ({type(e).__name__}); using recorded fallback histories.")


## The engine: trace → score → reflect → accept-if-better

It is the **same agentic loop** from the rest of the course — run, look at the trace,
decide, repeat — except now it is pointed at the agent's *own prompt*:

1. **Run** the current prompt over the train set and collect traces.
2. **Score** them (here: classification accuracy).
3. **Reflect** in natural language on the *failures*.
4. **Mutate** the prompt from that reflection.
5. **Accept** the candidate only if held-out score improves.

Step 5 is the whole ballgame. Without it, step 4 just memorizes the train set.

### A transparent GEPA-style reflective optimizer

This is a faithful **toy** of the core idea in
[GEPA: *Reflective Prompt Evolution Can Outperform Reinforcement Learning*](https://arxiv.org/abs/2507.19457)
(Agrawal et al., arXiv 2507.19457; ICLR 2026 Oral): reflect on traces in natural
language, then mutate the prompt. GEPA's headline result is that reflective *text*
evolution beat a GRPO RL baseline by ~10% on average (up to 20%) using up to **35×
fewer rollouts** — the point for this room: you can get the gains in **text space**,
no gradient updates, no GPU. We are **not** reproducing those numbers; we show the loop.

We keep a tiny **Pareto set** of the best prompt seen per instance-bucket (a dict),
which is GEPA's trick for not collapsing onto one global optimum too early.

**The live classifier.** `live_classify` is a real model call routed through
`invoke_sync` (so the synchronous optimizer can call the async model). Two details that
*are* part of the adaptation lesson: we **memoize** on `(prompt, text)` so repeated
classifications across epochs are free, and we accumulate the real `cost_usd` in
`LIVE_STATS` — proof the call was live.

In [3]:
LIVE_CACHE = {}                              # (prompt, text) -> code  (repeats are free)
LIVE_STATS = {"calls": 0, "cost_usd": 0.0}   # real cost: evidence the call was live

def live_classify(prompt, text):
    """Real model call: classify one line into a scrap code. Memoized + cost-tracked.

    The *prompt* IS the trainable parameter — we send it verbatim as the system prompt.
    A weak prompt (bare code names) leaves the model guessing on the confusable lines;
    the disambiguation rules reflection appends are what actually lift the score. We add
    only a thin output-format reminder so we can parse a single code back."""
    sys_p = prompt + "\n\nReply with EXACTLY one code from: " + ", ".join(CODES)
    key = (sys_p, text)
    if key in LIVE_CACHE:
        return LIVE_CACHE[key]
    resp = invoke_sync(model, [Message(role="system", content=sys_p),
                               Message(role="user", content="Scrap line: " + text)])
    LIVE_STATS["calls"] += 1
    LIVE_STATS["cost_usd"] += (resp.cost_usd or 0.0)
    out = (resp.content or "").strip().upper()
    code = next((c for c in CODES if c in out), CODES[0])
    LIVE_CACHE[key] = code
    return code


**The scorer and the offline rubric.** A prompt is scored by running it over examples
and measuring accuracy. The live "hero" run uses `live_classify`; the heavier A/B and
skill-doc sweeps use a transparent keyword **rubric** so their convergence *shape* is
stable in any room. A mutated prompt changes the rubric score by injecting `HINT:` lines
the rubric reads — that is how a prompt edit moves the number.

**The live/offline switch and the one-line classifier.** `USE_LIVE_MODEL` is flipped on only around the bounded hero run below; otherwise `classify_one` falls back to the transparent rubric so every visual still renders offline.

In [ ]:
# USE_LIVE_MODEL is toggled True only around the bounded hero run below.
USE_LIVE_MODEL = False

def classify_one(prompt, text):
    """Label one line: live model call when USE_LIVE_MODEL, else the transparent rubric."""
    if USE_LIVE_MODEL and MODEL_AVAILABLE:
        return live_classify(prompt, text)
    return rubric_label(prompt, text)

**The offline rubric — one keyword set per code.** This is the transparent judge that keeps the convergence *shape* stable in any room. A mutated prompt earns points by injecting extra hint keywords the rubric reads.

In [ ]:
# The offline-robust judge. A "prompt" can inject extra hint keywords per code, which is
# how a mutated prompt actually changes the score.
RUBRIC = {
    "DIM_OOT":   ["tolerance", "oversize", "undersize", "diameter", "flatness", "spec", "mm", "gd&t", "deviation"],
    "SURF_DEF":  ["scratch", "blemish", "scuff", "pitting", "finish", "cosmetic", "orange-peel", "burr"],
    "MAT_FLAW":  ["inclusion", "void", "porosity", "lamination", "billet", "material", "stock", "hard spot"],
    "MACH_SET":  ["fixture", "offset", "alignment", "setup", "datum", "tailstock", "jaw", "locating"],
    "TOOL_WEAR": ["insert", "dull", "worn", "tool wear", "reamer", "end mill", "chatter", "tap", "drill"],
    "OP_ERR":    ["operator", "wrong program", "misload", "skipped", "overrode", "backwards", "routing"],
    "CONTAM":    ["coolant", "debris", "oil", "grit", "contamination", "chip", "residue"],
}

**Score one line with the rubric.** Pick the code with the most keyword hits; hints added by a mutated prompt count double, so a good edit moves the number.

In [ ]:
def rubric_label(prompt, text):
    t = text.lower()
    extra = _hints_from_prompt(prompt)        # mutated prompt adds discriminating hints
    best, best_score = "DIM_OOT", -1
    for code, kws in RUBRIC.items():
        score = sum(1 for k in kws if k in t) + sum(2 for k in extra.get(code, []) if k in t)
        if score > best_score:
            best, best_score = code, score
    return best

**Read the hints a mutated prompt injected.** Parse any `HINT: <code> -> kw, kw` lines so the rubric can reward them. This is the mechanism by which a *prompt edit* changes the *score*.

In [ ]:
def _hints_from_prompt(prompt):
    """Parse 'HINT: <code> -> kw, kw' lines a mutation may add to the prompt."""
    hints = {}
    for line in prompt.splitlines():
        if line.strip().upper().startswith("HINT:"):
            body = line.split(":", 1)[1]
            if "->" in body:
                code, kws = body.split("->", 1)
                code = code.strip().upper()
                if code in CODES:
                    hints[code] = [k.strip().lower() for k in kws.split(",") if k.strip()]
    return hints

**Run a prompt over a set and score it.** `run_and_trace` labels one example and records whether it was right; `scorer` runs the whole set and returns accuracy plus the traces we reflect on.

In [ ]:
def run_and_trace(prompt, ex):
    pred = classify_one(prompt, ex["text"])
    return {"text": ex["text"], "gold": ex["label"], "pred": pred, "correct": pred == ex["label"]}

def scorer(prompt, examples):
    traces = [run_and_trace(prompt, ex) for ex in examples]
    return sum(t["correct"] for t in traces) / len(traces), traces

**The reflective mutation and the GEPA step.** This is the heart of GEPA: take the
*actual failures*, reflect on them, and rewrite the prompt.

- **Live path** (`reflect_live`): the model **reads its own wrong answers** and writes
  one short disambiguation rule that would have fixed them — a real NL reflection,
  appended to the prompt. This is what makes the live convergence curve climb.
- **Offline path** (`reflect_rubric`): a deterministic stand-in that appends a `HINT:`
  line the keyword rubric reads — same *mechanism*, no network, stable shape.

A `gepa_step` runs the prompt, collects failures, and returns the mutated candidate. The
**Pareto set** keeps the best prompt seen per gold code.

**Offline reflection — append a discriminating `HINT:` line.** Bucket the failures by gold code, pull the keywords that actually appeared in the missed lines, and add them as a hint the rubric reads next epoch.

In [ ]:
def reflect_rubric(prompt, failures):
    """Offline: append a discriminating HINT line the keyword rubric reads."""
    from collections import Counter
    miss = Counter(f["gold"] for f in failures)        # bucket failures by gold code
    target = miss.most_common(1)[0][0]
    missed_texts = " ".join(f["text"].lower() for f in failures if f["gold"] == target)
    kws = [k for k in RUBRIC[target] if k in missed_texts][:4] or RUBRIC[target][:2]
    hint = f"HINT: {target} -> {', '.join(kws)}"
    if hint in prompt:                                 # already learned; nudge another code
        for alt in miss:
            alt_kws = [k for k in RUBRIC[alt] if k in missed_texts][:3] or RUBRIC[alt][:2]
            alt_hint = f"HINT: {alt} -> {', '.join(alt_kws)}"
            if alt_hint not in prompt:
                return prompt.rstrip() + "\n" + alt_hint
    return prompt.rstrip() + "\n" + hint

**Live reflection — the model reads its OWN mistakes and writes a rule.** This is the genuine GEPA move: show the model the lines it got wrong, ask for one disambiguation rule, append it to the prompt for the next epoch.

In [ ]:
def reflect_live(prompt, failures):
    """Live: the model reads its OWN mistakes and writes one disambiguation rule.

    This is a genuine reflective mutation — the rule is generated from the actual
    misclassifications, then appended to the prompt for the next epoch."""
    shown = "\n".join(
        f"- line: {f['text']}\n  I answered {f['pred']}, correct was {f['gold']}"
        for f in failures[:6])
    reflect_sys = (
        "You tune a scrap-code classifier prompt. Below are lines it got WRONG, with the "
        "wrong code it gave and the correct code. Write ONE short rule (max 25 words, "
        "starting 'RULE:') that disambiguates the codes it confused, so it stops making "
        "these mistakes. Output only the rule line.")
    resp = invoke_sync(model, [Message(role="system", content=reflect_sys),
                               Message(role="user", content=shown)])
    LIVE_STATS["calls"] += 1
    LIVE_STATS["cost_usd"] += (resp.cost_usd or 0.0)
    rule = (resp.content or "").strip().splitlines()[0].strip()
    if not rule:
        return prompt
    if not rule.upper().startswith("RULE:"):
        rule = "RULE: " + rule
    if rule in prompt:                                 # already learned this exact rule
        return prompt
    return prompt.rstrip() + "\n" + rule

**Pick the reflection path, then take one GEPA step.** `reflect_and_mutate` routes to the live or offline path; `gepa_step` runs the prompt, collects the failures, and returns the mutated candidate.

In [ ]:
def reflect_and_mutate(prompt, failures):
    """Reflect on failures, propose a sharper prompt. Live model when available."""
    if not failures:
        return prompt
    if USE_LIVE_MODEL and MODEL_AVAILABLE:
        return reflect_live(prompt, failures)
    return reflect_rubric(prompt, failures)

def gepa_step(prompt, examples):
    _, traces = scorer(prompt, examples)               # RUN + collect traces
    failures = [t for t in traces if not t["correct"]]
    return reflect_and_mutate(prompt, failures)        # reflect -> reflective mutation

**Keep a Pareto set — the best prompt seen per code.** GEPA's trick for not collapsing onto one global optimum too early: remember the prompt that did best on each gold code.

In [ ]:
# Pareto set: best prompt per instance-bucket (here: per gold code).
pareto = {}
def update_pareto(prompt, traces):
    from collections import defaultdict
    by_code = defaultdict(list)
    for t in traces:
        by_code[t["gold"]].append(t["correct"])
    for code, oks in by_code.items():
        acc = sum(oks) / len(oks)
        if code not in pareto or acc > pareto[code][1]:
            pareto[code] = (prompt, acc)

print("optimizer ready:", OPTIMIZER, "| gate:", USE_HELDOUT_GATE, "| model live:", MODEL_AVAILABLE)

### Run the optimization across epochs (with the gate)

Watch the prompt text *get more specific* each epoch — it learns the edge cases from
its own failure traces. The **accept-if-better** gate is the only thing that decides
whether a candidate sticks. The hero run scores every line with the **live model** on a
bounded subset; the printed `cost_usd` is proof the calls were real.

**The optimize loop — and the accept-if-better gate.** Each epoch: mutate the prompt, score it on train and held-out, and accept the candidate only if the held-out score improves (or, with the gate off, if train improves). That one `if` is the whole lesson.

In [ ]:
def optimize(seed_prompt, epochs, use_gate, examples_train, examples_val):
    """Returns (best_prompt, history); history rows are (epoch, train, val, accepted)."""
    prompt = seed_prompt
    base_train, base_traces = scorer(prompt, examples_train)
    base_val, _ = scorer(prompt, examples_val)
    update_pareto(prompt, base_traces)
    best_train, best_val = base_train, base_val
    history = [(-1, round(base_train, 3), round(base_val, 3), True)]   # epoch -1 = seed
    for epoch in range(epochs):
        cand = gepa_step(prompt, examples_train)
        train_score, traces = scorer(cand, examples_train)
        val_score, _        = scorer(cand, examples_val)
        accept = (val_score > best_val) if use_gate else (train_score > best_train)
        if accept:
            prompt = cand
            best_val = max(best_val, val_score)
            best_train = max(best_train, train_score)
            update_pareto(prompt, traces)
        history.append((epoch, round(train_score, 3), round(val_score, 3), accept))
    return prompt, history


**Run the live hero loop.** A weak seed prompt classifies, reflects on its own misses, appends a rule, and we accept only if held-out improves. Train/val are bounded so the live pass fits the time budget; `live_classify` is memoized so repeats are free.

In [ ]:
# ── HERO RUN: a real, live GEPA loop ──
# Weak seed prompt -> the model classifies, reflects on its OWN misses, appends a
# disambiguation rule, and we accept only if held-out improves. Bounded train/val so the
# live pass fits the time budget; live_classify is memoized so repeats are free.
random.seed(SEED)
if MODEL_AVAILABLE:
    LIVE_TRAIN = train[:15]
    LIVE_VAL   = heldout[:15]
    USE_LIVE_MODEL = True
    best_prompt, history = optimize(SEED_PROMPT, EPOCHS, USE_HELDOUT_GATE, LIVE_TRAIN, LIVE_VAL)
    USE_LIVE_MODEL = False
    PATH_TAKEN = "live model"
    print(f"[live] {LIVE_STATS['calls']} real model calls, "
          f"cumulative cost_usd = ${LIVE_STATS['cost_usd']:.5f}\n")
else:
    best_prompt, history = optimize(SEED_PROMPT, EPOCHS, USE_HELDOUT_GATE, train, heldout)
    PATH_TAKEN = "offline rubric"

# If the model is unreachable the curve is whatever the deterministic rubric
# produced — shown as-is. We never paste a recorded/hand-authored shape over it.
val_series = [h[2] for h in history]
if not MODEL_AVAILABLE and max(val_series) - min(val_series) < 1e-6:
    print("[offline] held-out curve is flat on the rubric path — run on a host with a "
          "live model (ANTHROPIC_API_KEY / LiteLLM in env) to see the real convergence.")


**Read the result.** Watch the prompt text get *more specific* and the held-out score climb, with each epoch's accept/reject decision printed.

In [ ]:
print(f"path taken: {PATH_TAKEN}")
val_gain = history[-1][2] - history[0][2]
print(f"held-out: seed={history[0][2]:.3f} -> final={history[-1][2]:.3f}  "
      f"(gain {val_gain:+.3f})\n")
print("evolving prompt (final):\n" + "-" * 60)
print(best_prompt)
print("-" * 60)
print("\nepoch  train   val   accepted")
for ep, tr, va, ac in history:
    tag = "seed" if ep < 0 else str(ep)
    print(f"  {tag:>4}  {tr:>5}  {va:>5}   {'accept' if ac else 'reject'}")

### Visual 1 — convergence: held-out score by epoch (the hero)

A prompt (plain text) getting measurably better from its own failure traces.

**Said honestly:** the *shape* of this curve is model- and seed-dependent. A small
local model may improve in fits and starts or plateau early. The accept-if-better gate
guarantees the held-out curve is **monotonic non-decreasing**, but *not* that it climbs
fast. The teaching point is the **mechanism** (reflect → mutate → gate), not a number.

**One call into the chart helper.** Two curves over epochs — the train score and the held-out score — from the history table above.

In [ ]:
lines({"train score": [h[1] for h in history],
       "held-out (val) score": [h[2] for h in history]},
      x=[h[0] for h in history],
      title="Convergence — prompt improving from its own failures",
      xlabel="epoch  (-1 = seed prompt)", ylabel="accuracy")

## Now remove the gate

Set `USE_HELDOUT_GATE = False` in the control cell and re-run, or just run the A/B
below — it runs **both arms live** (on a host with a model) and plots the real
result. The gate is the only thing that changes between the two arms:

- **Gated** accepts a candidate only when **held-out** improves, so its held-out
  curve is monotonic non-decreasing by construction.
- **Ungated** accepts any candidate that improves **train**, so it will keep
  editing the prompt to fit the training lines even when that stops helping —
  held-out then **stalls or falls**.

How wide the gap opens is model- and seed-dependent; the point is the *mechanism*.
An ungated self-edit that looks good only on the data it was tuned on is, in
production, indistinguishable from
[context poisoning](https://www.dbreunig.com/2025/06/22/how-contexts-fail-and-how-to-fix-them.html)
injected into every future call.

### A/B contrast — gate vs no-gate

We run both configurations from a single config list and collect both histories, so the
comparison is one loop rather than two near-identical cells.

**Run both configs from one list.** Same seed, same seed prompt — only the gate differs — so the comparison is one loop, not two near-identical cells. If a rubric run stays flat we fall back to the recorded shape.

In [8]:
AB_CONFIGS = [
    {"name": "gated",   "use_gate": True},
    {"name": "ungated", "use_gate": False},
]

# Run BOTH arms the SAME way the hero run does — live on the host, deterministic
# rubric offline. No recorded shapes are pasted over the result: the curves below
# are whatever this run actually produced. Bounded train/val keeps the live A/B
# inside the time budget; live_classify is memoized so repeats are free.
AB_TRAIN, AB_VAL = (train[:15], heldout[:15]) if MODEL_AVAILABLE else (train, heldout)

ab_histories = {}
for cfg in AB_CONFIGS:
    random.seed(SEED)                       # same seed -> same starting conditions
    pareto.clear()
    USE_LIVE_MODEL = MODEL_AVAILABLE        # live model on the host; rubric offline
    _, hist = optimize(SEED_PROMPT, EPOCHS, cfg["use_gate"], AB_TRAIN, AB_VAL)
    USE_LIVE_MODEL = False
    ab_histories[cfg["name"]] = hist
    print(f"{cfg['name']:>8}: final held-out = {hist[-1][2]:.3f}  (train = {hist[-1][1]:.3f})")

AB_PATH = "live model" if MODEL_AVAILABLE else "offline rubric (deterministic)"
print(f"\n[A/B ran on: {AB_PATH}] — real result, nothing recorded.")


   gated: final held-out = 0.800  (train = 0.880)
 ungated: final held-out = 0.450  (train = 0.950)


### Visual 2 — gate vs no-gate (read the real curves)

Three lines from the live A/B above: **gated held-out**, **ungated held-out**, and
the **ungated train** line the ungated arm kept chasing. The gated held-out can
only hold or climb; watch whether the ungated held-out keeps up with its own train
line or drifts below it. When it drifts, that gap *is* overfitting —
*without a held-out gate, "self-improving" is just "self-drifting."*

**One call into the chart helper.** Three curves over epochs: the gated held-out, the ungated held-out, and the ungated *train* line it kept chasing while held-out drifted down.

In [ ]:
gated   = ab_histories["gated"]
ungated = ab_histories["ungated"]
lines({"gated — held-out":          [h[2] for h in gated],
       "ungated — held-out":        [h[2] for h in ungated],
       "ungated — train (chased)":  [h[1] for h in ungated]},
      x=[h[0] for h in ungated],
      title="Gate vs no-gate — the held-out gate is what makes it 'improving'",
      xlabel="epoch  (-1 = seed)", ylabel="accuracy")

## The cheapest lever — write a lesson back into the identity file

Before any optimizer, the simplest adaptation: when a run fails, **reflect on why in one
line and write it down** — into memory now, and if it is durable, promote it into the
`identity.md` the agent loads on every future run. That is
[Reflexion](https://arxiv.org/abs/2303.11366) made operational: reflect → store → do
better next trial, with **no training run** anywhere.

We do it on a **real failure trace** from the GEPA run above: take a line the prompt got
wrong, have the model write a one-line `**Lesson:**`, and append it to a real identity
file's `## Lessons learned` section. The diff that ships is plain Markdown a reviewer
signs off on — the same promotion pattern the identity section used
(`sarah_chen__prelesson.md` → `sarah_chen.md`).

**Get a real failure trace.** Prefer one from the live GEPA seed run; fall back to a recorded trace offline. This is the raw material the lesson is reflected from.

In [ ]:
# ── LESSONS-LEARNED LEVER: a real failure trace -> a one-line lesson -> the identity file ──
# 1) Get a real failure. Prefer one from the GEPA hero run; else a recorded trace.
seed_traces = scorer(SEED_PROMPT, heldout)[1]            # the WEAK seed's traces (real)
failures = [t for t in seed_traces if not t["correct"]]
if failures:
    fail = failures[0]
    fail_src = "live GEPA seed run"
else:                                                    # offline safety net
    fail = {"text": "fixture shifted mid-run, bore walked 0.3mm off datum",
            "gold": "MACH_SET", "pred": "DIM_OOT"}
    fail_src = "recorded trace"

print(f"failure trace ({fail_src}):")
print(f"  line:   {fail['text']!r}")
print(f"  agent answered {fail['pred']}, correct was {fail['gold']}")

**Reflect the failure into one durable lesson.** Live: the model writes a one-line lesson that would prevent this class of mistake. Offline: a deterministic stand-in does the same.

In [ ]:
# 2) Reflect the failure into ONE durable lesson line (live model, else deterministic).
def reflect_lesson(fail):
    if MODEL_AVAILABLE:
        sys_p = ("An agent misclassified a scrap line. Write ONE durable lesson (max 25 "
                 "words) that would prevent this class of mistake on future runs. "
                 "Output only the lesson sentence, no preamble.")
        usr = (f"line: {fail['text']}\nit answered {fail['pred']}, correct was {fail['gold']}")
        resp = invoke_sync(model, [Message(role="system", content=sys_p),
                                   Message(role="user", content=usr)])
        LIVE_STATS["calls"] += 1
        LIVE_STATS["cost_usd"] += (resp.cost_usd or 0.0)
        line = (resp.content or "").strip().splitlines()[0].strip().strip('"')
        if line:
            return line
    # deterministic fallback
    return (f"A line describing {fail['gold'].replace('_',' ').lower()} conditions is "
            f"{fail['gold']}, not {fail['pred']} — read the cause, not just the symptom.")

lesson = reflect_lesson(fail)
import datetime
today = datetime.date.today().isoformat()
lesson_entry = f"- {today} — *auto-promoted from trace:* {lesson} **Lesson durable -> promoted.**"
print("the lesson written:")
print(f"  {lesson_entry}")

**Append the lesson under `## Lessons learned`.** Start from the pre-lesson identity file and insert the entry into the right section, creating it if missing.

In [ ]:
# 3) Append it to a REAL identity file's "## Lessons learned" section (before -> after diff).
ID_DIR = DATA.parent / "identities"
src = ID_DIR / "sarah_chen__prelesson.md"                # start from the pre-lesson version
identity_before = src.read_text() if src.exists() else (
    "---\nname: scrap-coder-agent\n---\n# scrap-coder-agent\n\n"
    "## Lessons learned (auto-promoted from past traces)\n")

def append_lesson(doc, entry):
    """Insert the lesson under the '## Lessons learned' header (create it if missing)."""
    header = "## Lessons learned"
    lines = doc.splitlines()
    for i, ln in enumerate(lines):
        if ln.startswith(header):
            j = i + 1
            while j < len(lines) and (lines[j].strip().startswith("-") or lines[j].strip()==""):
                j += 1
            lines.insert(j, entry)
            return "\n".join(lines)
    return doc.rstrip() + f"\n\n{header} (auto-promoted from past traces)\n{entry}\n"

identity_after = append_lesson(identity_before, lesson_entry)
out_path = ADAPT / "scrap_coder_identity.after.md"        # write to the adaptation workspace
out_path.write_text(identity_after)

**Show the diff that ships.** A readable one-line Markdown change a human reviews and merges — adaptation proposes, you dispose.

In [ ]:
# Show the one-line diff that ships — the whole point: a readable Markdown change.
print("identity diff (the change that ships):")
print("  --- before  (## Lessons learned)")
print("  +++ after")
print(f"  +{lesson_entry}")
print(f"\nwrote {out_path.relative_to(DATA.parent)}  "
      f"({len(identity_before.split())} -> {len(identity_after.split())} words)")
print("a human reviews this diff and merges it — adaptation proposes, you dispose.")

## Same engine, different artifact — train the *skill doc*

Now the trainable parameter is a `SKILL.md`. Same loop — trace → score → propose an
edit → accept only if held-out improves — but the "edit" is a bounded
**add / delete / replace** on the skill's Knowledge / Antipatterns sections.

A **textual learning rate** caps how far the doc can drift in one step, and a
**meta-skill** check rejects edits that bloat the doc past a word budget. This mirrors
Microsoft's [*SkillOpt: Executive Strategy for Self-Evolving Agent Skills*](https://arxiv.org/abs/2605.23904)
(arXiv 2605.23904): text-space, validation-gated, and **zero added inference-time model
calls at deployment** — the optimized skill is just a better static document.

That last property is the lab sell: the gain ships as a **reviewable Markdown diff**,
so it goes through the same change-control / audit path as any other config file —
not as new weights.

**Optimize the skill doc.** We start from a small `scrap-coder` SKILL.md, propose a
bounded edit each epoch, and accept only when held-out improves **and** the doc stays
under the word budget and within the per-step learning rate. The accepted edits are the
diff that ships.

**The seed skill doc and its guardrails.** A small `scrap-coder` SKILL.md whose Knowledge / Antipatterns sections are trainable, plus a word budget (the meta-skill) and a per-step textual learning rate that caps drift.

In [ ]:
# A small scrap-coder SKILL.md (the artifact we optimize). Knowledge + Antipatterns
# are the trainable sections; the rest is held fixed.
SEED_SKILL = dedent("""    ---
    name: scrap-coder
    version: 1.0.0
    description: Classify a scrap-log line into one of seven scrap codes.
    ---
    # scrap-coder

    ## Knowledge
    - Codes: DIM_OOT, SURF_DEF, MAT_FLAW, MACH_SET, TOOL_WEAR, OP_ERR, CONTAM.
    - Read the line, pick the single best-fitting code.

    ## Antipatterns
    - Do not invent codes outside the seven.
    """)

WORD_BUDGET = 320          # meta-skill: reject edits that bloat past this
LEARNING_RATE_WORDS = 60   # textual learning rate: max words added per accepted step

def skill_word_count(doc):
    return len(doc.split())

def skill_to_prompt(doc):
    """Turn the skill's Knowledge HINT bullets into prompt lines the rubric scorer reads."""
    prompt = SEED_PROMPT
    for line in doc.splitlines():
        s = line.strip("- ").strip()
        if s.upper().startswith("HINT:"):
            prompt += "\n" + s
    return prompt

**Propose one bounded edit.** Reflect on the failures, find a fresh `HINT:` line, and insert it under `## Knowledge` — a single add/delete/replace, never a rewrite.

In [ ]:
def propose_skill_edit(doc, failures):
    """A bounded add/delete/replace edit. Returns (new_doc, action, edit_text)."""
    cand_prompt = reflect_and_mutate(skill_to_prompt(doc), failures)
    new_hints = [l for l in cand_prompt.splitlines() if l.strip().upper().startswith("HINT:")]
    existing = [l.strip("- ").strip() for l in doc.splitlines() if l.strip().strip("- ").upper().startswith("HINT:")]
    fresh = [h for h in new_hints if h not in existing]
    if not fresh:
        return doc, "noop", ""
    add = fresh[0]
    out, action = [], "add"                  # insert under ## Knowledge
    for line in doc.splitlines():
        out.append(line)
        if line.strip() == "## Knowledge":
            out.append(f"- {add}")
    return "\n".join(out), action, add        # edit_text = the line added

def skill_val_score(doc):
    return scorer(skill_to_prompt(doc), heldout)[0]

**The skill optimize loop.** Same trace → propose → gate shape as the prompt optimizer, but an edit is accepted only if held-out improves **and** it stays under the word budget **and** within the per-step learning rate.

In [ ]:
def optimize_skill(seed_doc, epochs):
    doc = seed_doc
    best_val = skill_val_score(doc)
    # traj rows: (epoch, action, word_count, val, accepted, word_delta, edit_text)
    traj = [(0, "seed", skill_word_count(doc), round(best_val, 3), True, 0, "")]
    for i in range(1, epochs + 1):
        _, traces = scorer(skill_to_prompt(doc), train)
        failures = [t for t in traces if not t["correct"]]
        cand, action, edit_text = propose_skill_edit(doc, failures)
        wc = skill_word_count(cand)
        delta = wc - skill_word_count(doc)
        val = skill_val_score(cand)
        within_budget = wc <= WORD_BUDGET                       # meta-skill budget
        within_lr = delta <= LEARNING_RATE_WORDS                # textual learning rate
        accept = (val > best_val) and within_budget and within_lr   # held-out gate
        if accept:
            doc = cand
            best_val = val
        traj.append((i, action, wc, round(val, 3), accept, delta, edit_text))
    return doc, traj

**Run the skill optimizer.** Bounded train/val sweep; if the rubric trajectory stays flat we load the recorded SkillOpt shape so the picture still lands.

In [ ]:
random.seed(SEED)
best_skill, skill_traj = optimize_skill(SEED_SKILL, EPOCHS)

# Offline-robust: if the rubric trajectory is flat, use the recorded SkillOpt shape.
if len({t[3] for t in skill_traj}) <= 1:
    fb = json.loads((ADAPT / "skill_opt_fallback.json").read_text())
    # pad recorded rows to the (…, delta, edit_text) shape
    rows = fb["trajectory"]
    skill_traj = []
    prev_wc = rows[0][2]
    for r in rows:
        delta = r[2] - prev_wc
        edit = r[5] if len(r) > 5 else ""
        skill_traj.append((r[0], r[1], r[2], r[3], r[4], delta, edit))
        prev_wc = r[2]
    WORD_BUDGET = fb["word_budget"]
    print("[offline] using recorded SkillOpt trajectory")

(ADAPT / "best_skill.md").write_text(best_skill)             # write the optimized skill

**Read the bounded edits.** Each accepted change is small and bounded, and you can read it — the diff a reviewer signs off on.

In [ ]:
# The point of this slide: each accepted change is SMALL and BOUNDED, and you can read it.
print(f"bounded edits  (learning rate <= {LEARNING_RATE_WORDS} words/step, "
      f"budget <= {WORD_BUDGET} words):\n")
for i, action, wc, val, acc, delta, edit in skill_traj:
    if i == 0:
        continue
    status = "ACCEPTED" if acc else "rejected"
    reason = "" if acc else ("  <- over budget" if wc > WORD_BUDGET
                             else "  <- over learning-rate" if delta > LEARNING_RATE_WORDS
                             else "  <- no held-out gain")
    sign = f"{delta:+d}"
    print(f"  edit {i}: {action:>7}  dwords={sign:>4}  words={wc:>3}/{WORD_BUDGET}  "
          f"held-out={val:.3f}  {status}{reason}")
    if edit:
        verb = "+ added" if action == "add" else ("- removed" if action == "delete" else "~ changed")
        print(f"           {verb}: {edit[:88]}")
print(f"\nbest_skill.md written ({skill_word_count(best_skill)} words) — small steps, "
      f"each a readable diff a reviewer can sign off.")

### Visual 3 — skill score vs word-count (bloat control)

Each accepted edit plotted as (word_count, held-out score). The meta-skill keeps the
doc **bounded under the word budget** while the score rises — the skill stays "readable
in a few minutes." Rejected edits (e.g. ones that bloat past budget with no gain) are
shown so you can see the gate working.

**One call into the chart helper.** Each edit as a point of (word count, held-out score), labeled by epoch so you can read the trajectory — the score rises while the word count stays bounded under the budget.

In [ ]:
scatter({f"e{t[0]} {t[1]}{'' if t[4] else ' (rej)'}": (t[2], t[3]) for t in skill_traj},
        title="SkillOpt — score rises, word-count stays bounded (budget %d words)" % WORD_BUDGET,
        xlabel="skill doc word count", ylabel="held-out accuracy")

## Graduate a trace into a reusable tool (autobrowse-style)

The agent found a reliable path. **Freeze it into a deterministic tool** — the
push-down move, done automatically.

This is the "graduate a converged trajectory into a durable, reusable artifact" idea
from [ReUseIt: *Synthesizing Reusable AI Agent Workflows for Web Automation*](https://arxiv.org/abs/2510.14308)
(arXiv 2510.14308) and Browserbase's [Autobrowse](https://www.browserbase.com/blog/autobrowse).
We do it offline on a genuinely **deterministic sub-procedure** — which is the honest
case: graduate the part that is *actually* deterministic into code; leave the
genuinely-judgment parts to the model.

**State the model-dependence aloud:** token/step savings here are real and measurable,
but the magnitude depends on how repetitive the task is. This is a worked illustration,
not a benchmark.

**Manufacture the tool, then prove it.** We write the repeated sub-procedure (a
per-code count rollup) to `scripts/scrap_rollup.py`, register it as an arcrun `Tool`,
and hand it to the agentic loop — the model **calls** the graduated tool instead of
counting by hand. The before/after step and token numbers come from the recorded
illustration.

**Manufacture the tool.** Write the repeated deterministic sub-procedure — a per-code count rollup — to `scripts/scrap_rollup.py`. This is the part that is *actually* deterministic, so it earns the right to be frozen into code.

In [ ]:
# The agent kept doing the same deterministic sub-procedure: total scrap counts per
# code. We graduate it into a real script under scripts/ and register it as a Tool.
SCRIPTS_OUT.mkdir(parents=True, exist_ok=True)
rollup_path = SCRIPTS_OUT / "scrap_rollup.py"
rollup_path.write_text(dedent('''\
    """Graduated tool: deterministic scrap-count rollup.

    Manufactured from a converged agent trajectory in NB07 (adaptation).
    Replaces N per-line LLM "add this up" turns with one deterministic call.
    """
    import json
    from collections import Counter


    def rollup(labeled_lines: list[dict]) -> dict:
        """Count scrap lines per code. Same output every time, given same input."""
        counts = Counter(r["label"] for r in labeled_lines)
        return {"total": len(labeled_lines), "by_code": dict(sorted(counts.items()))}


    if __name__ == "__main__":
        import sys
        rows = [json.loads(l) for l in open(sys.argv[1]) if l.strip()]
        print(json.dumps(rollup(rows), indent=2))
    '''))

**Register it as a tool the loop can call.** Import the freshly written module and wrap `rollup` as an arcrun `Tool`, so the model invokes it instead of counting by hand.

In [ ]:
# import + register as a Tool
sys.path.insert(0, str(SCRIPTS_OUT))
import scrap_rollup
importlib.reload(scrap_rollup)

async def scrap_rollup_tool_fn(args: dict, ctx: ToolContext) -> str:
    rows = json.loads(args["rows_json"])
    return json.dumps(scrap_rollup.rollup(rows))

scrap_rollup_tool = Tool(
    name="scrap_rollup",
    description="Deterministically count labeled scrap lines per code. "
                "Input: rows_json = JSON list of {text,label}. Returns {total, by_code}.",
    input_schema={"type": "object",
                  "properties": {"rows_json": {"type": "string"}},
                  "required": ["rows_json"]},
    execute=scrap_rollup_tool_fn,
)

# Sanity: the graduated tool produces the right answer.
truth = scrap_rollup.rollup(LINES)
print("graduated tool output:", json.dumps(truth))

**Prove the graduation.** Hand the registered tool to the loop and let the model **call** it — one deterministic tool turn replaces the per-line counting. We use `run_sync` so the loop lands on the same background loop the model client is bound to.

In [ ]:
# PROVE the graduation: hand the registered tool to the loop and let the model CALL it
# (one deterministic tool turn replaces the per-line "add this up" turns). We run the
# agent loop via run_sync so it lands on the SAME background loop the model client is
# bound to (top-level await would use the kernel loop and break the client binding).
if MODEL_AVAILABLE:
    sample = LINES[:12]
    task = ("Total up these labeled scrap rows by code using the scrap_rollup tool, "
            "then report how many rows there are in total.\nrows: " + json.dumps(sample))

    grad_res = run_sync(run(model, StaticProvider([scrap_rollup_tool]),
                            "You are a scrap-reporting agent. Use the scrap_rollup tool to "
                            "count rows; do not count by hand.",
                            task, max_turns=4))
    print(f"[live] agent called the graduated tool: tool_calls_made="
          f"{grad_res.tool_calls_made}, turns={grad_res.turns}")
    print("agent answer:", (grad_res.content or "").strip().splitlines()[0][:100])

**Load the before/after numbers.** The recorded illustration: the agent used to do the rollup as several LLM turns; now it is one deterministic call — fewer steps and tokens, same answer.

In [ ]:
# before = recorded baseline (the agent doing the rollup as LLM turns, pre-graduation);
# after  = measured from THIS run\'s live tool call when the model is reachable.
fb = json.loads((ADAPT / "graduation_fallback.json").read_text())
before = fb["before"]   # recorded baseline — labeled as such below

if MODEL_AVAILABLE:
    after = {"llm_steps": grad_res.turns, "tokens": None, "accuracy": before["accuracy"]}
    after_src = "measured live"
else:
    after = fb["after"]
    after_src = "recorded baseline"

aft_tok = f"{after['tokens']} tokens" if after.get("tokens") is not None else "tokens n/a (live)"
print(f"before graduation (recorded baseline): {before['llm_steps']} LLM steps, "
      f"{before['tokens']} tokens, acc={before['accuracy']}")
print(f"after  graduation ({after_src}): {after['llm_steps']} LLM steps, "
      f"{aft_tok}, acc={after['accuracy']}")
print(f"\nwrote {rollup_path.relative_to(DATA.parent)} and registered tool 'scrap_rollup'")


### Visual 4 — before/after graduation (steps + tokens, accuracy held constant)

The system manufactured its own script. LLM steps and tokens drop; accuracy is
unchanged because the graduated part was deterministic all along.

**One call into the chart helper.** The two metrics that move — LLM steps and tokens — before vs after graduation. Accuracy is unchanged (the graduated part was deterministic all along), so it stays out of the bars and in the title.

In [ ]:
# Steps are real when live; token magnitudes come from the recorded baseline (a true
# before/after token count needs two full live runs). Chart title says which is which.
_btok = before["tokens"]; _atok = (after["tokens"] if after.get("tokens") is not None else fb["after"]["tokens"])
compare({"steps · before":  before["llm_steps"],
         "steps · after":   after["llm_steps"],
         "tokens · before": _btok,
         "tokens · after":  _atok},
        title="Trace -> tool graduation — steps %s; token magnitudes are a recorded baseline" %
              ("measured live" if MODEL_AVAILABLE else "recorded"),
        ylabel="count")

## Fine-tuning, framed (no GPU in the room)

What you'd do *with volume*: curate good decision traces (the audit log from earlier
sections), fine-tune a local open-weights model **inside the security
boundary, on an on-prem GPU**, version the model, repeat.

We **don't** run it live — there's no GPU here, and a real fine-tune would dwarf the
25-minute budget. But the on-ramp is just data prep, so we show *that* step.

**Text-first, weights-last.** Every cell above got gains in **text space**, as
reviewable diffs, with no GPU. Fine-tuning is the *last* lever you reach for — it's the
most expensive to run, the hardest to audit, and the easiest to overfit. For a lab the
on-prem angle matters: text-space optimization works against a hosted API *or* a fully
air-gapped local model, while weight updates require the data **and** the GPUs to be
in-boundary.

### Assemble a fine-tune dataset from the audit log

> *The audit log you built for compliance is the dataset you optimize from.*

Filter the audit/trace store to **good-outcome** decision traces and write a JSONL
fine-tune file in the standard chat format. No training is run.

**Filter the audit log to good-outcome traces and write a fine-tune JSONL.** The audit log you built for compliance *is* the dataset — we keep only the good outcomes and format them as standard chat records. No training is run here.

In [ ]:
audit_rows = [json.loads(l) for l in (ADAPT / "audit_traces.jsonl").read_text().splitlines() if l.strip()]
good = [r for r in audit_rows if r["outcome"] == "good"]

ft_path = ADAPT / "finetune_scrap.jsonl"
with ft_path.open("w") as f:
    for r in good:
        record = {"messages": [
            {"role": "system", "content": "Classify the scrap-log line into one scrap code."},
            {"role": "user", "content": r["input"]},
            {"role": "assistant", "content": r["output"]},
        ]}
        f.write(json.dumps(record) + "\n")

print(f"audit traces: {len(audit_rows)} total, {len(good)} good-outcome (kept)")
print(f"wrote {ft_path.relative_to(DATA.parent)}  ({len(good)} records)\n")
print("sample fine-tune records:")
for line in ft_path.read_text().splitlines()[:2]:
    print(json.dumps(json.loads(line), indent=2))
print("\nNext step (NOT run here): fine-tune an on-prem open-weights model on this file,")
print("version it, and route through the same audit path as any other artifact.")

## ✅ TODO — optimize your own prompt

Drop in a weak prompt for one of *your* tasks plus a tiny labeled set, split it into train / val, and run the **gated** optimizer for a few epochs. Keep the diff only if held-out improves.

In [15]:
# ✅ TODO — replace with your own weak prompt and labeled examples.
MY_SEED_PROMPT = "Label each item."                       # ✅ TODO: your weak prompt

MY_DATA = [                                               # ✅ TODO: your tiny labeled set
    {"text": "example item one", "label": "LABEL_A"},
    {"text": "example item two", "label": "LABEL_B"},
    # ... add ~12+ rows so a train/val split is meaningful
]

# ✅ TODO: a scorer for YOUR task. The keyword-rubric here is a stand-in — swap in your
# own (or rely on the live model path) and define what "correct" means for your labels.
MY_VALID_LABELS = sorted({r["label"] for r in MY_DATA})

if len(MY_DATA) >= 4:
    random.seed(SEED)
    rows = MY_DATA[:]
    random.shuffle(rows)
    cut = max(1, int(len(rows) * 0.6))
    my_train, my_val = rows[:cut], rows[cut:]
    print(f"your split: {len(my_train)} train / {len(my_val)} val")
    print("✅ TODO: wire MY_VALID_LABELS into RUBRIC (or use the live model), then call")
    print("        optimize(MY_SEED_PROMPT, EPOCHS, USE_HELDOUT_GATE, my_train, my_val)")
    print("        and keep best_prompt only if the held-out score went up.")
else:
    print("✅ TODO: add your own labeled rows to MY_DATA above, then re-run.")


✅ TODO: add your own labeled rows to MY_DATA above, then re-run.


## Takeaway

> **Adaptation is the agentic loop pointed at the agent itself:** trace → score →
> propose an edit → **accept only if held-out improves.** Optimize **text before
> weights**, and keep a human on the merge.

- The cheapest lever needs no optimizer at all: **reflect on a failure and write the
  lesson back** — into memory, and into the `identity.md` if it is durable. That is
  Reflexion, and it is a readable Markdown diff a human signs off.
- The held-out gate is the line between **self-improving** and **self-drifting**. An
  ungated self-edit is indistinguishable from poisoning every future call.
- The same engine optimizes a **prompt**, a **skill doc** (with **bounded** add / delete
  / replace edits under a word budget), or graduates a trace into a **tool** — all in
  text space, all shipping as reviewable diffs, no GPU required.
- Fine-tuning is the last lever, not the first: most expensive, hardest to audit,
  easiest to overfit. And the audit log you built for compliance *is* the dataset.

One section left — finding the work worth pointing all of this at.